# **PONTIFICIA UNIVERSIDAD JAVERIANA**
## **Procesamiento de alto volumen de datos**
**Fecha:** 7 de Abril del 2026

**Autor:** Grupo Sigma

**Tema:** Proyecto de Big Data

**Objetivo:** 
- Entender la importancia del uso de herramientas de Big Data en entornos empresariales, a fin de poder solucionar preguntas de negocio.
- Entender el paso a paso de un proyecto de procesamiento de datos para la generación de hallazgos de valor basado en la metodología CRISP-DM.
- Documentar la implementación de un cluster como infraestructura de procesamiento de grandes volúmenes de datos, a través de máquinas virtuales.
- Realizar procesamiento de datos aplicado a los resultados de las pruebas Saber 11 del ICFES a nivel nacional.

**Dataset:** Resultados_únicos_Saber_11_20260408.csv  
**Fuente:** Instituto Colombiano para la Evaluación de la Educación (ICFES) — Datos Abiertos  
**Registros:** ~7.11 millones de filas | 51 columnas  

**Version:** Entrega 1

Para asegurar que el proyecto funcione correctamente con pandas, matplotlib, seaborn y findspark, ejecutar el siguiente comando desde la raíz del proyecto
```bash
pip install -r requirements.txt
```

In [1]:
### Importación de bibliotecas basicas 
import os                       # -> Para gestion de archivos y procesos
import sys                      # -> Para manejo de recursos del sistema
import pandas as pd             # -> Para graficar y objetos dataframe
import numpy as np              # -> Para algebra matricial
import matplotlib.pyplot as plt # -> Para formatos de graficas
import seaborn as sns           # -> Para estadistica y graficar
import scipy.stats as stats     # -> Para pruebas estadisticas

In [2]:
### Importacion de bibliotecas especializadas
import findspark                                # -> Para manejo del entorno de PySpark
findspark.init('/Almacen/Spark')                # -> Se inicia el entorno para PySpark
from pyspark import SparkConf, SparkContext     # -> Para contexto y configuración de PySpark
from pyspark.sql import SparkSession            # -> Para manejo de Sesion en entorno de consultas SQL
from pyspark.sql.functions import *             # -> Para funciones de manipulacion de columnas
from pyspark.sql.types import IntegerType, StringType, DoubleType # -> Para definir tipos de datos
import pyspark.sql.functions as F               # -> Para funciones de manipulacion de columnas (alias)
from pyspark.ml.feature import VectorAssembler  # -> Para construcción de vectores  
from pyspark.ml.stat import Correlation         # -> Para calculo de correlaciones

In [3]:
configura = SparkConf()
configura.set('spark.scheduler.mode', 'FAIR')
configura.set('spark.scheduler.allocation', '/Almacen/Spark/conf/fairscheduler.xml')
configura.setMaster('spark://10.43.97.166:7077')
configura.setAppName('SigmaSPARK_ICFES')

sparkSigma = SparkSession.builder.config(conf=configura).getOrCreate()
sparkSigma

## **Lectura de data**

In [4]:
# Lectura del archivo CSV 

dfPy00 = sparkSigma.read.format("csv").option("header","true").load("../data/Resultados_únicos_Saber_11_20260408.csv")
dfPy00.show(5)

+-------+------------------+----------------+-------------------+-------------+---------------+-----------------+-----------------------------+------------------+------------------------+------------------------+-----------------+--------------------+-----------+------------+--------------------+---------------+---------------------------+----------------+-------------------+---------------------------+---------------------------+---------------------+---------------------+-----------------------+-----------------+------------------------+---------------+--------------------+-----------+-----------------------+-----------------+-----------------+----------------+---------------------+-----------------+--------------------+--------------------+--------------------+------------------+-------------------+--------------------+------------------+------------------+-------------+-----------+----------------+------------------------+----------------+--------------------+-----------+
|PERIODO|

### 1 - Descripción general del dataset

El dataset corresponde a los **Resultados únicos de las pruebas Saber 11** publicados por el **ICFES (Instituto Colombiano para la Evaluación de la Educación)**. Contiene información sobre el desempeño académico de estudiantes de grado 11 en Colombia, junto con variables socioeconómicas y de contexto familiar.

**Principales grupos de variables:**
- **Identificación y período:** Año, periodo, número de documento del estudiante.
- **Ubicación geográfica:** Departamento y municipio del colegio y de residencia.
- **Contexto socioeconómico:** Estrato, nivel educativo de los padres, acceso a internet, computador en casa, número de personas en el hogar.
- **Información del colegio:** Nombre, naturaleza (oficial/privado), calendario, área (urbano/rural).
- **Puntajes por área:** Lectura crítica, matemáticas, ciencias naturales, sociales y ciudadanas, inglés.
- **Puntaje global:** Suma ponderada de las áreas evaluadas.

In [5]:
NombresOriginales = [
    # Estudiante
    'ESTU_TIPODOCUMENTO', 'ESTU_CONSECUTIVO', 'ESTU_GENERO', 'ESTU_FECHANACIMIENTO',
    'ESTU_NACIONALIDAD', 'ESTU_PAIS_RESIDE', 'ESTU_PRIVADO_LIBERTAD', 'ESTU_ESTUDIANTE',
    'ESTU_ESTADOINVESTIGACION',
    # Ubicación estudiante
    'ESTU_COD_DEPTO_PRESENTACION', 'ESTU_COD_MCPIO_PRESENTACION',
    'ESTU_COD_RESIDE_DEPTO', 'ESTU_COD_RESIDE_MCPIO',
    'ESTU_DEPTO_PRESENTACION', 'ESTU_MCPIO_PRESENTACION',
    'ESTU_DEPTO_RESIDE', 'ESTU_MCPIO_RESIDE',
    # Colegio - características
    'COLE_AREA_UBICACION', 'COLE_BILINGUE', 'COLE_CALENDARIO', 'COLE_CARACTER',
    'COLE_GENERO', 'COLE_JORNADA', 'COLE_NATURALEZA', 'COLE_SEDE_PRINCIPAL',
    # Colegio - nombres
    'COLE_NOMBRE_ESTABLECIMIENTO', 'COLE_NOMBRE_SEDE',
    # Colegio - códigos
    'COLE_COD_DANE_ESTABLECIMIENTO', 'COLE_COD_DANE_SEDE',
    'COLE_COD_DEPTO_UBICACION', 'COLE_COD_MCPIO_UBICACION', 'COLE_CODIGO_ICFES',
    # Colegio - ubicación
    'COLE_DEPTO_UBICACION', 'COLE_MCPIO_UBICACION',
    # Familia
    'FAMI_ESTRATOVIVIENDA', 'FAMI_TIENEINTERNET', 'FAMI_TIENECOMPUTADOR',
    'FAMI_TIENEAUTOMOVIL', 'FAMI_TIENELAVADORA',
    'FAMI_EDUCACIONPADRE', 'FAMI_EDUCACIONMADRE',
    'FAMI_NUMLIBROS', 'FAMI_CUARTOSHOGAR', 'FAMI_PERSONASHOGAR',
    # Puntajes
    'DESEMP_INGLES', 'PUNT_INGLES', 'PUNT_MATEMATICAS', 'PUNT_C_NATURALES',
    'PUNT_SOCIALES_CIUDADANAS', 'PUNT_LECTURA_CRITICA', 'PUNT_GLOBAL',
    # Periodo
    'PERIODO',
]

NombresNuevos = [
    # Estudiante
    'TIPO_DOC', 'ID_ESTUDIANTE', 'GENERO', 'FECHA_NAC',
    'NACIONALIDAD', 'PAIS', 'PRIVADO_LIBERTAD', 'TIPO_ESTUDIANTE',
    'ESTADO_INVESTIGACION',
    # Ubicación estudiante
    'COD_DEPTO_PRESENTACION', 'COD_MUN_PRESENTACION',
    'COD_DEPTO_RESIDENCIA', 'COD_MUN_RESIDENCIA',
    'DEPTO_PRESENTACION', 'MUN_PRESENTACION',
    'DEPTO_RESIDENCIA', 'MUN_RESIDENCIA',
    # Colegio - características
    'AREA_COLEGIO', 'BILINGUE', 'CALENDARIO', 'CARACTER_COLEGIO',
    'GENERO_COLEGIO', 'JORNADA', 'NATURALEZA', 'SEDE_PRINCIPAL',
    # Colegio - nombres
    'NOMBRE_COLEGIO', 'NOMBRE_SEDE',
    # Colegio - códigos
    'COD_DANE_COLEGIO', 'COD_DANE_SEDE',
    'COD_DEPTO_COLEGIO', 'COD_MUN_COLEGIO', 'COD_ICFES',
    # Colegio - ubicación
    'DEPTO_COLEGIO', 'MUN_COLEGIO',
    # Familia
    'ESTRATO', 'TIENE_INTERNET', 'TIENE_COMPUTADOR',
    'TIENE_AUTOMOVIL', 'TIENE_LAVADORA',
    'EDU_PADRE', 'EDU_MADRE',
    'NUM_LIBROS', 'CUARTOS_HOGAR', 'PERSONAS_HOGAR',
    # Puntajes
    'DESEMP_INGLES', 'PUNT_INGLES', 'PUNT_MATE', 'PUNT_CIENCIAS',
    'PUNT_SOCIALES', 'PUNT_LECTURA', 'PUNT_GLOBAL',
    # Periodo
    'PERIODO',
]

# Aplicar renombres
dfPy01 = dfPy00
for antes, nuevo in zip(NombresOriginales, NombresNuevos):
    if antes in dfPy01.columns:
        dfPy01 = dfPy01.withColumnRenamed(antes, nuevo)

# Verificación
print(f'Columnas en dfPy00: {len(dfPy00.columns)}')
print(f'Columnas en dfPy01: {len(dfPy01.columns)}')
print(f'Columnas sin renombrar (mantienen nombre original):')
sin_renombrar = [c for c in dfPy01.columns if c in dfPy00.columns and c not in NombresNuevos]
print(sin_renombrar if sin_renombrar else 'Ninguna')

Columnas en dfPy00: 51
Columnas en dfPy01: 51
Columnas sin renombrar (mantienen nombre original):
Ninguna


### 2 - Tipos y coherencia de datos

In [6]:
dfPy01.printSchema()

root
 |-- PERIODO: string (nullable = true)
 |-- TIPO_DOC: string (nullable = true)
 |-- ID_ESTUDIANTE: string (nullable = true)
 |-- AREA_COLEGIO: string (nullable = true)
 |-- BILINGUE: string (nullable = true)
 |-- CALENDARIO: string (nullable = true)
 |-- CARACTER_COLEGIO: string (nullable = true)
 |-- COD_DANE_COLEGIO: string (nullable = true)
 |-- COD_DANE_SEDE: string (nullable = true)
 |-- COD_DEPTO_COLEGIO: string (nullable = true)
 |-- COD_MUN_COLEGIO: string (nullable = true)
 |-- COD_ICFES: string (nullable = true)
 |-- DEPTO_COLEGIO: string (nullable = true)
 |-- GENERO_COLEGIO: string (nullable = true)
 |-- JORNADA: string (nullable = true)
 |-- MUN_COLEGIO: string (nullable = true)
 |-- NATURALEZA: string (nullable = true)
 |-- NOMBRE_COLEGIO: string (nullable = true)
 |-- NOMBRE_SEDE: string (nullable = true)
 |-- SEDE_PRINCIPAL: string (nullable = true)
 |-- COD_DEPTO_PRESENTACION: string (nullable = true)
 |-- COD_MUN_PRESENTACION: string (nullable = true)
 |-- COD_DE

In [7]:
# ── dfPy02: Transformación de tipos de dato ───────────────────────────────────

dfPy02 = dfPy01

# ── 1. Enteros directos (puntajes y período) ──────────────────────────────────

COLS_INT = ['PUNT_INGLES', 'PUNT_MATE', 'PUNT_CIENCIAS',
            'PUNT_SOCIALES', 'PUNT_LECTURA', 'PUNT_GLOBAL', 'PERIODO']

for c in COLS_INT:
    if c in dfPy02.columns:
        dfPy02 = dfPy02.withColumn(c, F.col(c).cast('int'))

# ── 2. Fecha ──────────────────────────────────────────────────────────────────

dfPy02 = dfPy02.withColumn('FECHA_NAC',
    F.to_date(F.col('FECHA_NAC'), 'dd/MM/yyyy')
)

# ── 3. ESTRATO: extraer número de "Estrato 1", "Estrato 2", etc. ──────────────

dfPy02 = dfPy02.withColumn('ESTRATO',
    F.regexp_extract(F.col('ESTRATO'), r'(\d+)', 1).cast('int')
)

# ── 4. CUARTOS_HOGAR: texto → double ─────────────────────────────────────────

def mapear_cuartos(col_name):
    c = F.upper(F.trim(F.col(col_name)))
    return (
        F.when(c == 'UNO',                          1.0)
         .when(c == 'DOS',                          2.0)
         .when(c == 'TRES',                         3.0)
         .when(c == 'CUATRO',                       4.0)
         .when(c == 'CINCO',                        5.0)
         .when(c == 'SEIS',                         6.0)
         .when(c == 'SIETE',                        7.0)
         .when(c == 'OCHO',                         8.0)
         .when(c == 'NUEVE',                        9.0)
         .when(c.isin('SEIS O MAS', 'SEIS O MÁS'), 6.0)
         .when(c.isin('DIEZ O MÁS', 'DIEZ O MAS'), 10.0)
         .otherwise(F.lit(None).cast('double'))
    )

dfPy02 = dfPy02.withColumn('CUARTOS_HOGAR', mapear_cuartos('CUARTOS_HOGAR'))

# ── 5. PERSONAS_HOGAR: texto y rangos → double (punto medio para rangos) ──────

def mapear_personas(col_name):
    c = F.upper(F.trim(F.col(col_name)))
    return (
        F.when(c == 'UNA',                          1.0)
         .when(c == 'DOS',                          2.0)
         .when(c == 'TRES',                         3.0)
         .when(c == 'CUATRO',                       4.0)
         .when(c == 'CINCO',                        5.0)
         .when(c == 'SEIS',                         6.0)
         .when(c == 'SIETE',                        7.0)
         .when(c == 'OCHO',                         8.0)
         .when(c == 'NUEVE',                        9.0)
         .when(c == 'DIEZ',                        10.0)
         .when(c == 'ONCE',                        11.0)
         .when(c.isin('DOCE O MÁS', 'DOCE O MAS'), 12.0)
         .when(c == '1 A 2',                        1.5)
         .when(c == '3 A 4',                        3.5)
         .when(c == '5 A 6',                        5.5)
         .when(c == '7 A 8',                        7.5)
         .when(c.isin('9 O MÁS', '9 O MAS'),        9.0)
         .otherwise(F.lit(None).cast('double'))
    )

dfPy02 = dfPy02.withColumn('PERSONAS_HOGAR', mapear_personas('PERSONAS_HOGAR'))

# ── 6. Estandarización de texto en categóricas (upper + trim) ─────────────────

COLS_STRING = [
    # Estudiante
    'GENERO', 'NACIONALIDAD', 'PAIS', 'PRIVADO_LIBERTAD',
    'TIPO_ESTUDIANTE', 'ESTADO_INVESTIGACION', 'TIPO_DOC',
    # Ubicación
    'DEPTO_PRESENTACION', 'MUN_PRESENTACION', 'DEPTO_RESIDENCIA', 'MUN_RESIDENCIA',
    # Colegio
    'AREA_COLEGIO', 'BILINGUE', 'CALENDARIO', 'CARACTER_COLEGIO',
    'GENERO_COLEGIO', 'JORNADA', 'NATURALEZA', 'SEDE_PRINCIPAL',
    'DEPTO_COLEGIO', 'MUN_COLEGIO',
    # Familia
    'TIENE_INTERNET', 'TIENE_COMPUTADOR', 'TIENE_AUTOMOVIL', 'TIENE_LAVADORA',
    'EDU_PADRE', 'EDU_MADRE', 'NUM_LIBROS',
    # Puntaje inglés
    'DESEMP_INGLES',
]

for c in COLS_STRING:
    if c in dfPy02.columns:
        dfPy02 = dfPy02.withColumn(c, F.upper(F.trim(F.col(c))))

# ── Verificación ──────────────────────────────────────────────────────────────

print('Schema final dfPy02:')
dfPy02.printSchema()

Schema final dfPy02:
root
 |-- PERIODO: integer (nullable = true)
 |-- TIPO_DOC: string (nullable = true)
 |-- ID_ESTUDIANTE: string (nullable = true)
 |-- AREA_COLEGIO: string (nullable = true)
 |-- BILINGUE: string (nullable = true)
 |-- CALENDARIO: string (nullable = true)
 |-- CARACTER_COLEGIO: string (nullable = true)
 |-- COD_DANE_COLEGIO: string (nullable = true)
 |-- COD_DANE_SEDE: string (nullable = true)
 |-- COD_DEPTO_COLEGIO: string (nullable = true)
 |-- COD_MUN_COLEGIO: string (nullable = true)
 |-- COD_ICFES: string (nullable = true)
 |-- DEPTO_COLEGIO: string (nullable = true)
 |-- GENERO_COLEGIO: string (nullable = true)
 |-- JORNADA: string (nullable = true)
 |-- MUN_COLEGIO: string (nullable = true)
 |-- NATURALEZA: string (nullable = true)
 |-- NOMBRE_COLEGIO: string (nullable = true)
 |-- NOMBRE_SEDE: string (nullable = true)
 |-- SEDE_PRINCIPAL: string (nullable = true)
 |-- COD_DEPTO_PRESENTACION: string (nullable = true)
 |-- COD_MUN_PRESENTACION: string (nullab

In [8]:
# Análisis de Frecuencias para Variables Categóricas

# 1. Distribución por Género
print('Distribución por GÉNERO:')
dfPy02.groupby(['GENERO']).count().show()

# 2. Distribución por Área del colegio
print('Distribución por ÁREA DEL COLEGIO:')
dfPy02.groupby(['AREA_COLEGIO']).count().show()

# 3. Distribución por Naturaleza del colegio
print('Distribución por NATURALEZA DEL COLEGIO:')
dfPy02.groupby(['NATURALEZA']).count().show()

# 4. Distribución por Jornada
print('Distribución por JORNADA:')
dfPy02.groupby(['JORNADA']).count().show()

# 5. Distribución por Estrato
print('Distribución por ESTRATO:')
dfPy02.groupby(['ESTRATO']).count().orderBy('ESTRATO').show()

# 6. Distribución por Acceso a Internet
print('Distribución por TIENE_INTERNET:')
dfPy02.groupby(['TIENE_INTERNET']).count().show()

# Validación de Integridad de la Carga
print('-' * 30)
print('TOTAL DE REGISTROS:', dfPy02.count())
print('TOTAL REGISTROS ÚNICOS (DISTINCT):', dfPy02.distinct().count())
print('-' * 30)

# Verificación de nulos en llave primaria
print('REGISTROS SIN ID_ESTUDIANTE (NULOS):')
print(dfPy02.filter(F.col('ID_ESTUDIANTE').isNull()).count())

Distribución por GÉNERO:
+------+-------+
|GENERO|  count|
+------+-------+
|     F|3880677|
|  NULL|   3406|
|     M|3225621|
+------+-------+

Distribución por ÁREA DEL COLEGIO:
+------------+-------+
|AREA_COLEGIO|  count|
+------------+-------+
|        NULL|   3031|
|       RURAL|1000710|
|      URBANO|6105963|
+------------+-------+

Distribución por NATURALEZA DEL COLEGIO:
+----------+-------+
|NATURALEZA|  count|
+----------+-------+
|      NULL|     23|
|   OFICIAL|5134719|
|NO OFICIAL|1974962|
+----------+-------+

Distribución por JORNADA:
+--------+-------+
| JORNADA|  count|
+--------+-------+
|   UNICA| 464190|
|  MAÑANA|3413717|
|    NULL|     23|
|SABATINA| 454343|
|   TARDE| 964261|
|COMPLETA|1346879|
|   NOCHE| 466291|
+--------+-------+

Distribución por ESTRATO:
+-------+-------+
|ESTRATO|  count|
+-------+-------+
|   NULL| 307552|
|      1|2488247|
|      2|2441528|
|      3|1348516|
|      4| 323398|
|      5| 125691|
|      6|  74772|
+-------+-------+

Distribu

In [9]:
# Función para validar si es numérico y decidir si usar isnan
def count_missings(df):
    output = []
    for c, dtype in df.dtypes:
        # Si la columna es Double o Float, evaluar isNull e isnan
        if dtype in ['double', 'float']:
            output.append(F.count(F.when(F.col(c).isNull() | F.isnan(F.col(c)), c)).alias(c))
        # Para el resto, solo isNull
        else:
            output.append(F.count(F.when(F.col(c).isNull(), c)).alias(c))
    return output

# Ejecutar el select con la lógica filtrada
dfPy02.select(count_missings(dfPy02)).show()

+-------+--------+-------------+------------+--------+----------+----------------+----------------+-------------+-----------------+---------------+---------+-------------+--------------+-------+-----------+----------+--------------+-----------+--------------+----------------------+--------------------+--------------------+------------------+------------------+----------------+--------------------+---------------+---------+------+----------------+--------------+------------+----+----------------+-------------+---------+---------+-------+--------------+---------------+----------------+--------------+--------------+-------------+-----------+---------+-------------+-------------+------------+-----------+
|PERIODO|TIPO_DOC|ID_ESTUDIANTE|AREA_COLEGIO|BILINGUE|CALENDARIO|CARACTER_COLEGIO|COD_DANE_COLEGIO|COD_DANE_SEDE|COD_DEPTO_COLEGIO|COD_MUN_COLEGIO|COD_ICFES|DEPTO_COLEGIO|GENERO_COLEGIO|JORNADA|MUN_COLEGIO|NATURALEZA|NOMBRE_COLEGIO|NOMBRE_SEDE|SEDE_PRINCIPAL|COD_DEPTO_PRESENTACION|COD_MUN_

In [10]:
# Cálculo de porcentajes de nulos
import pyspark.sql.functions as F

total_registros = dfPy02.count()

print(f'Cantidad total de registros: {total_registros}')

columnas_a_revisar = [
    'GENERO', 'ESTRATO', 'PUNT_GLOBAL', 'PUNT_MATE',
    'PUNT_LECTURA', 'PUNT_CIENCIAS', 'PUNT_SOCIALES', 'PUNT_INGLES',
    'EDU_PADRE', 'EDU_MADRE', 'TIENE_INTERNET', 'TIENE_COMPUTADOR'
]

for col_name in columnas_a_revisar:
    if col_name in dfPy02.columns:
        nulos = dfPy02.filter(F.col(col_name).isNull()).count()
        pct = nulos * 100 / total_registros
        print(f'Porcentaje registros nulos en {col_name}: {__builtins__.round(pct, 4)}%')

duplicados = total_registros - dfPy02.distinct().count()

print(f'\nCantidad de registros duplicados: {duplicados}')
print(f'Porcentaje registros duplicados: {__builtins__.round(duplicados * 100 / total_registros, 4)}%')

Cantidad total de registros: 7109704
Porcentaje registros nulos en GENERO: 0.0479%
Porcentaje registros nulos en ESTRATO: 4.3258%
Porcentaje registros nulos en PUNT_GLOBAL: 36.7037%
Porcentaje registros nulos en PUNT_MATE: 0.0001%
Porcentaje registros nulos en PUNT_LECTURA: 36.7037%
Porcentaje registros nulos en PUNT_CIENCIAS: 36.7037%
Porcentaje registros nulos en PUNT_SOCIALES: 36.7037%
Porcentaje registros nulos en PUNT_INGLES: 0.0589%
Porcentaje registros nulos en EDU_PADRE: 2.9878%
Porcentaje registros nulos en EDU_MADRE: 2.9935%
Porcentaje registros nulos en TIENE_INTERNET: 2.6952%
Porcentaje registros nulos en TIENE_COMPUTADOR: 2.1269%

Cantidad de registros duplicados: 1387111
Porcentaje registros duplicados: 19.5101%


### **Analisis de nulos**

Tras ejecutar el diagnóstico de frecuencias y conteo de vacíos, se observan los siguientes hallazgos:
- **Género:** Variable con cobertura casi completa. Los registros nulos representan una fracción mínima y serán eliminados para no afectar los análisis demográficos.
- **Puntajes (PUNT_*):** Las columnas de puntajes son las variables objetivo del análisis. Los valores nulos corresponden a presentaciones incompletas o inválidas, y serán excluidos del análisis de rendimiento.
- **Educación de padres (EDU_PADRE, EDU_MADRE):** Variables con porcentaje relevante de valores nulos o categoría `No sabe`. Son importantes para el análisis socioeconómico.
- **Estrato:** Presenta valores nulos y posibles inconsistencias en la codificación (valores fuera del rango 1-6). Se tratarán como categoría especial `SIN_DATO`.
- **Internet y Computador:** Variables binarias con posibles nulos, relevantes para medir la brecha de conectividad digital.
- **Duplicados:** Se detectarán registros duplicados para aplicar la función `.distinct()` antes del modelado.

### **Decisiones de Tratamiento**

- **Limpieza de duplicados:** Se aplicará `.distinct()` para conservar una única presentación por estudiante por período.
- **Puntajes nulos:** Se eliminarán los registros donde `PUNT_GLOBAL` sea nulo, ya que representan presentaciones incompletas o inválidas.
- **Género nulo:** Se eliminarán los 105 registros con género NULL dada su irrelevancia estadística.
- **Estrato:** Los valores fuera del rango 1-6 se reclasificarán como `SIN_DATO` para no distorsionar el análisis socioeconómico.
- **Consolidación de Categorías (Educación de padres):** Se implementará un mapeo para unificar las categorías similares en etiquetas estándar.
- **Exclusión de Variables Incompletas:** Las columnas `PAIS` y `FECHA_NAC` serán descartadas de los análisis predictivos por su bajo aporte informativo.

## **Exploración de los datos**

In [11]:
# Estadísticas descriptivas para las variables numéricas principales
dfPy02.select(
    'ESTRATO', 'PUNT_LECTURA', 'PUNT_MATE', 'PUNT_CIENCIAS',
    'PUNT_SOCIALES', 'PUNT_INGLES', 'PUNT_GLOBAL'
).summary().show()

+-------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|summary|           ESTRATO|      PUNT_LECTURA|         PUNT_MATE|     PUNT_CIENCIAS|     PUNT_SOCIALES|       PUNT_INGLES|       PUNT_GLOBAL|
+-------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|  count|           6802152|           4500181|           7109697|           4500181|           4500181|           7105518|           4500181|
|   mean| 2.026936475397786| 52.18493945021322| 49.26133349986645|50.106060178468375|48.865357860050516|  48.5413650348926|252.30331691103092|
| stddev|1.0467480262138757|10.385771971965873|11.904398229288253|10.569585208236283| 11.73055591533721|12.222449561920369|50.426683241030624|
|    min|                 1|                 0|                 0|                 0|                 0|                -1|                 0|

A continuación se realiza una exploración visual de las principales variables numéricas del dataset para identificar su distribución, presencia de valores atípicos y posibles transformaciones necesarias.

In [ ]:
# Conversión a Pandas para visualización
pdf = dfPy02.select(
    'ESTRATO', 'PUNT_LECTURA', 'PUNT_MATE', 'PUNT_CIENCIAS',
    'PUNT_SOCIALES', 'PUNT_INGLES', 'PUNT_GLOBAL',
    'GENERO', 'AREA_COLEGIO', 'NATURALEZA', 'TIENE_INTERNET'
).dropna(subset=['PUNT_GLOBAL']).toPandas()

print(f'Registros para visualización: {len(pdf)}')

In [ ]:
# Gráfico 1: Distribución del Puntaje Global (Histograma + Boxplot)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(pdf['PUNT_GLOBAL'].dropna(), bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribución del Puntaje Global Saber 11', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Puntaje Global')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(pdf['PUNT_GLOBAL'].mean(), color='red', linestyle='--', linewidth=1.5,
                label=f'Media: {pdf["PUNT_GLOBAL"].mean():.1f}')
axes[0].axvline(pdf['PUNT_GLOBAL'].median(), color='orange', linestyle='--', linewidth=1.5,
                label=f'Mediana: {pdf["PUNT_GLOBAL"].median():.1f}')
axes[0].legend()

# Boxplot
axes[1].boxplot(pdf['PUNT_GLOBAL'].dropna(), patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Boxplot Puntaje Global Saber 11', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Puntaje Global')
axes[1].set_xticks([1])
axes[1].set_xticklabels(['Puntaje Global'])

plt.tight_layout()
plt.show()

El histograma del Puntaje Global muestra una distribución aproximadamente normal con leve asimetría hacia la derecha. La mayoría de los estudiantes se concentra en el rango medio, con la media y la mediana cercanas entre sí. El boxplot permite identificar la presencia de valores atípicos tanto en el extremo inferior como en el superior, correspondientes a presentaciones de alto o bajo rendimiento.

In [ ]:
# Gráfico 2: Distribución de puntajes por área
puntajes_cols = ['PUNT_LECTURA', 'PUNT_MATE', 'PUNT_CIENCIAS', 'PUNT_SOCIALES', 'PUNT_INGLES']
colores = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']

fig, axes = plt.subplots(1, 5, figsize=(18, 5))

for i, (col, color) in enumerate(zip(puntajes_cols, colores)):
    data = pdf[col].dropna()
    axes[i].hist(data, bins=40, color=color, edgecolor='white', alpha=0.85)
    axes[i].set_title(col.replace('PUNT_', '').replace('_', ' ').title(), fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Puntaje')
    axes[i].set_ylabel('Frecuencia' if i == 0 else '')
    axes[i].axvline(data.mean(), color='red', linestyle='--', linewidth=1.2)

plt.suptitle('Distribución de Puntajes por Área — Saber 11', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

Los puntajes por área presentan distribuciones unimodales con leve variabilidad entre áreas. Inglés tiende a mostrar mayor dispersión y asimetría, mientras que Lectura Crítica y Matemáticas presentan distribuciones más simétricas. La línea roja indica la media de cada área.

In [ ]:
# Gráfico 4: Puntaje Global por Género
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

generos_validos = ['M', 'F']
pdf_genero = pdf[pdf['GENERO'].isin(generos_validos)]

for genero, color in zip(generos_validos, ['#4C72B0', '#DD8452']):
    data = pdf_genero[pdf_genero['GENERO'] == genero]['PUNT_GLOBAL'].dropna()
    axes[0].hist(data, bins=40, alpha=0.65, color=color, label=genero, edgecolor='white')

axes[0].set_title('Distribución Puntaje Global por Género', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Puntaje Global')
axes[0].set_ylabel('Frecuencia')
axes[0].legend(title='Género')

conteo = pdf_genero['GENERO'].value_counts()
axes[1].bar(conteo.index, conteo.values, color=['#4C72B0', '#DD8452'], edgecolor='white', alpha=0.85)
axes[1].set_title('Cantidad de Estudiantes por Género', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Género')
axes[1].set_ylabel('Cantidad de Estudiantes')
for j, (idx, val) in enumerate(zip(conteo.index, conteo.values)):
    axes[1].text(j, val + 1000, f'{val:,}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

La distribución del puntaje global por género muestra diferencias sutiles entre estudiantes masculinos y femeninos. El dataset está relativamente balanceado en términos de participación por género. Cualquier diferencia en el rendimiento medio debe interpretarse en conjunto con variables de contexto como el tipo de colegio y el estrato.

In [ ]:
# Gráfico 6: Puntaje Global según acceso a Internet
fig, ax = plt.subplots(figsize=(9, 5))

for cat, color in zip(['SI', 'NO'], ['#2196F3', '#FF7043']):
    data = pdf[pdf['TIENE_INTERNET'] == cat]['PUNT_GLOBAL'].dropna()
    if len(data) > 0:
        ax.hist(data, bins=40, alpha=0.65, color=color, label=f'Internet: {cat}', edgecolor='white')

ax.set_title('Distribución Puntaje Global según Acceso a Internet', fontsize=12, fontweight='bold')
ax.set_xlabel('Puntaje Global')
ax.set_ylabel('Frecuencia')
ax.legend()

plt.tight_layout()
plt.show()

Los estudiantes con acceso a internet en el hogar presentan una distribución de puntaje global desplazada hacia valores más altos, sugiriendo una posible asociación entre conectividad y rendimiento académico. Esta variable actúa como un proxy de recursos del hogar y será incluida en los modelos explicativos.

## **Planteamiento de sub-preguntas investigativas**

1. ¿Existe una brecha significativa en el puntaje global Saber 11 entre colegios oficiales y no oficiales, y cómo varía esta brecha según el departamento?
2. ¿Qué tan fuerte es la relación entre el nivel educativo de los padres (EDU_PADRE, EDU_MADRE) y el rendimiento académico del estudiante en cada área evaluada?
3. ¿Qué variables socioeconómicas (estrato, acceso a internet, computador en casa, número de personas en el hogar) son los mejores predictores del puntaje global en las pruebas Saber 11?
4. ¿Existen diferencias estadísticamente significativas en el rendimiento por área (matemáticas, inglés, ciencias) entre colegios urbanos y rurales a nivel nacional?

## **Limpieza, filtro y transformaciones iniciales**

In [ ]:
# Paso 1: Eliminación de duplicados
dfPy03 = dfPy02.distinct()
print('Registros tras eliminar duplicados:', dfPy03.count())

In [ ]:
# Paso 2: Eliminar registros sin PUNT_GLOBAL (presentaciones incompletas)
dfPy03 = dfPy03.filter(F.col('PUNT_GLOBAL').isNotNull())
print('Registros tras eliminar nulos en PUNT_GLOBAL:', dfPy03.count())

In [ ]:
# Paso 3: Eliminar registros sin GENERO
dfPy03 = dfPy03.filter(F.col('GENERO').isNotNull())
print('Registros tras eliminar nulos en GENERO:', dfPy03.count())

In [ ]:
# Filtrar solo registros post-2014, momento en el que se formulo el estandar actual de calificación del ICFES
dfPy03 = dfPy03.filter(F.col('PERIODO').substr(1, 4).cast('int') >= 2014)

print('Registros post-2014:', dfPy03.count())

In [ ]:
# Paso 4: Descarte de columnas con alta ausencia o bajo aporte informativo
columnas_a_descartar = ['PAIS', 'FECHA_NAC', 'TIPO_DOC']
dfPy03 = dfPy03.drop(*[c for c in columnas_a_descartar if c in dfPy03.columns])
print('Columnas restantes:', dfPy03.columns)

In [ ]:
# Paso 5: Ver todos los valores en ESTRATO
dfPy03.groupby('ESTRATO').count().orderBy('count', ascending=False).show(20, truncate=False)

In [ ]:
# Reclasificación del Estrato — valores fuera del rango 1-6 marcados como SIN_DATO
dfPy03 = dfPy03.withColumn('ESTRATO',
    F.when(F.col('ESTRATO').between(1, 6), F.col('ESTRATO').cast('string'))
     .otherwise(F.lit('SIN_DATO'))
)

print('Categorías de ESTRATO tras reclasificación:')
dfPy03.groupby('ESTRATO').count().orderBy(F.col('count').desc()).show()

In [ ]:
# Verificación final de nulos tras tratamiento
print('Conteo de nulos tras tratamiento:')
dfPy03.select(count_missings(dfPy03)).show()

print('Total de registros finales:', dfPy03.count())

# Clusters de estudiantes

Se buscará formar grupos de estudiantes diferentes que permitan conocer agrupaciónes que expliquen los tipos de estudiantes que presentan la prueba ICFES, con el fin de entender y desarrollar planes de acción dedicadas a estos grupos diferentes.

## Tratamiento de nulos en variables categóricas

Los nulos en variables socioeconómicas y de contexto se reemplazan por la categoría `"Sin dato"` para preservar todos los registros válidos en el clustering. Esta decisión evita sesgo por imputación en variables con ausencia informativa (por ejemplo, el estudiante o la familia no proporcionó el dato).

In [ ]:
COLS_SIN_DATO = [
    # Recursos del hogar
    'TIENE_INTERNET', 'TIENE_COMPUTADOR', 'TIENE_AUTOMOVIL', 'TIENE_LAVADORA',
    # Educación padres
    'EDU_PADRE', 'EDU_MADRE',
    # Colegio
    'JORNADA', 'AREA_COLEGIO', 'NATURALEZA', 'CALENDARIO',
    'BILINGUE', 'CARACTER_COLEGIO', 'GENERO_COLEGIO',
    # Inglés
    'DESEMP_INGLES',
]

for col_name in COLS_SIN_DATO:
    if col_name in dfPy03.columns:
        dfPy03 = dfPy03.withColumn(
            col_name,
            F.when(F.col(col_name).isNull(), 'Sin dato').otherwise(F.col(col_name))
        )

# Verificación
print('Nulos restantes en variables tratadas:')
dfPy03.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in COLS_SIN_DATO
]).show()

## Discretización de variables numéricas

Se convierten las variables numéricas a categorías ordinales para unificar el espacio de features antes del clustering con K-modes. Las decisiones de corte son:

| Variable | Criterio de corte | Niveles |
|---|---|---|
| `PUNT_MATE`, `PUNT_LECTURA`, `PUNT_CIENCIAS`, `PUNT_SOCIALES` | Niveles oficiales ICFES (0–40 / 41–55 / 56–70 / 71–100) | Insuficiente · Mínimo · Satisfactorio · Avanzado |
| `PUNT_INGLES` | Marco Común Europeo vía ICFES (0–47 / 48–57 / 58–67 / 68–78 / 79–100) | Pre-A1 · A1 · A2 · B1 · B+ |
| `PUNT_GLOBAL` | Cuartiles calculados por `PERIODO` | Bajo · Medio-bajo · Medio-alto · Alto |
| `HACINAMIENTO_CAT` | Terciles empíricos de `PERSONAS_HOGAR / CUARTOS_HOGAR` | Bajo · Medio · Alto · Sin dato |

Los cortes para las áreas e inglés son los reportados públicamente por el ICFES. Se recomienda verificarlos contra el documento técnico oficial de tu cohorte en caso de que el dataset incluya periodos anteriores a 2015.

Las columnas numéricas originales (`PUNT_*`, `PERSONAS_HOGAR`, `CUARTOS_HOGAR`) se conservan en el DataFrame para validación y perfilamiento posterior.

In [ ]:
from pyspark.sql import Window

# ── Cortes oficiales ICFES por área ──────────────────────────────────────────
# Fuente: niveles de desempeño publicados por ICFES (Insuficiente/Mínimo/Satisfactorio/Avanzado)
CORTES_AREA = {
    'PUNT_MATE':     (40, 55, 70),
    'PUNT_LECTURA':  (40, 55, 70),
    'PUNT_CIENCIAS': (40, 55, 70),
    'PUNT_SOCIALES': (40, 55, 70),
}

ETIQUETAS_AREA = ['1_Insuficiente', '2_Minimo', '3_Satisfactorio', '4_Avanzado']

# Cortes MCER para Inglés (ICFES Saber 11)
CORTES_INGLES = (47, 57, 67, 78)
ETIQUETAS_INGLES = ['1_PreA1', '2_A1', '3_A2', '4_B1', '5_Bmas']

# ── HACINAMIENTO_CAT ──────────────────────────────────────────────────────────

# Cast explícito a double para que approxQuantile funcione correctamente
dfPy03 = dfPy03.withColumn(
    'HACINAMIENTO_RAW',
    F.when(
        F.col('CUARTOS_HOGAR').isNotNull() &
        (F.col('CUARTOS_HOGAR') > 0) &
        F.col('PERSONAS_HOGAR').isNotNull(),
        (F.col('PERSONAS_HOGAR').cast('double') / F.col('CUARTOS_HOGAR').cast('double'))
    ).otherwise(F.lit(None).cast('double'))
)

# Verificación antes de calcular terciles
n_validos = dfPy03.filter(F.col('HACINAMIENTO_RAW').isNotNull()).count()
print(f'Registros válidos para calcular terciles: {n_validos:,}')

if n_validos == 0:
    raise ValueError('No hay registros válidos en HACINAMIENTO_RAW. '
                     'Revisar los tipos de CUARTOS_HOGAR y PERSONAS_HOGAR.')

# Calcular terciles sobre registros válidos
quantiles = dfPy03.filter(
    F.col('HACINAMIENTO_RAW').isNotNull()
).approxQuantile('HACINAMIENTO_RAW', [1/3, 2/3], 0.01)

q1, q2 = quantiles[0], quantiles[1]
print(f'Cortes de hacinamiento (terciles): Q1={q1:.2f}, Q2={q2:.2f}')

print(f'Cortes de hacinamiento (terciles): Q1={q1:.2f}, Q2={q2:.2f}')

# Categorizar
dfPy04 = dfPy03.withColumn(
    'HACINAMIENTO_CAT',
    F.when(F.col('HACINAMIENTO_RAW').isNull(), 'Sin dato')
     .when(F.col('HACINAMIENTO_RAW') <= q1,    '1_Bajo')
     .when(F.col('HACINAMIENTO_RAW') <= q2,    '2_Medio')
     .otherwise('3_Alto')
).drop('HACINAMIENTO_RAW')

# ── Puntajes por área → niveles ICFES ────────────────────────────────────────

for col_nombre, (c1, c2, c3) in CORTES_AREA.items():
    nivel_col = 'NIVEL_' + col_nombre.replace('PUNT_', '')
    dfPy04 = dfPy04.withColumn(
        nivel_col,
        F.when(F.col(col_nombre).isNull(),        'Sin dato')
         .when(F.col(col_nombre) <= c1,            ETIQUETAS_AREA[0])
         .when(F.col(col_nombre) <= c2,            ETIQUETAS_AREA[1])
         .when(F.col(col_nombre) <= c3,            ETIQUETAS_AREA[2])
         .otherwise(                               ETIQUETAS_AREA[3])
    )

# ── PUNT_INGLES → niveles MCER ────────────────────────────────────────────────

c1, c2, c3, c4 = CORTES_INGLES
dfPy04 = dfPy04.withColumn(
    'NIVEL_INGLES',
    F.when(F.col('PUNT_INGLES').isNull(),      'Sin dato')
     .when(F.col('PUNT_INGLES') <= c1,          ETIQUETAS_INGLES[0])
     .when(F.col('PUNT_INGLES') <= c2,          ETIQUETAS_INGLES[1])
     .when(F.col('PUNT_INGLES') <= c3,          ETIQUETAS_INGLES[2])
     .when(F.col('PUNT_INGLES') <= c4,          ETIQUETAS_INGLES[3])
     .otherwise(                                ETIQUETAS_INGLES[4])
)

print('Discretización de áreas e inglés completada.')
dfPy04.select(
    [c for c in dfPy04.columns if c.startswith('NIVEL_')]
).show(5)

In [ ]:
# ── PUNT_GLOBAL → cuartiles por PERIODO ──────────────────────────────────────
# Se usa ntile(4) particionando por PERIODO para que el cuartil sea relativo
# a la cohorte del estudiante, no a la distribución global de todos los años

window_periodo = Window.partitionBy('PERIODO').orderBy('PUNT_GLOBAL')

ETIQUETAS_GLOBAL = {
    '1': '1_Bajo',
    '2': '2_Medio_bajo',
    '3': '3_Medio_alto',
    '4': '4_Alto'
}

dfPy04 = dfPy04.withColumn(
    'NIVEL_GLOBAL_NUM',
    F.ntile(4).over(window_periodo)
).withColumn(
    'NIVEL_GLOBAL',
    F.col('NIVEL_GLOBAL_NUM').cast('string')
)

# Reemplazar números por etiquetas descriptivas
for num, etiqueta in ETIQUETAS_GLOBAL.items():
    dfPy04 = dfPy04.withColumn(
        'NIVEL_GLOBAL',
        F.when(F.col('NIVEL_GLOBAL') == num, etiqueta).otherwise(F.col('NIVEL_GLOBAL'))
    )

dfPy04 = dfPy04.drop('NIVEL_GLOBAL_NUM')

print('Distribución de NIVEL_GLOBAL por PERIODO (primeros 2 periodos):')
dfPy04.filter(
    F.col('PERIODO').isin(
        [r['PERIODO'] for r in dfPy04.select('PERIODO').distinct().limit(2).collect()]
    )
).groupBy('PERIODO', 'NIVEL_GLOBAL').count().orderBy('PERIODO', 'NIVEL_GLOBAL').show()

In [ ]:
# ── Resumen de variables discretizadas ───────────────────────────────────────

COLS_DISCRETIZADAS = [
    'NIVEL_MATE', 'NIVEL_LECTURA', 'NIVEL_CIENCIAS',
    'NIVEL_SOCIALES', 'NIVEL_INGLES', 'NIVEL_GLOBAL', 'HACINAMIENTO_CAT'
]

print(f'Total de registros en dfPy04: {dfPy04.count():,}')
print(f'Total de columnas: {len(dfPy04.columns)}\n')

for col_name in COLS_DISCRETIZADAS:
    print(f'─── {col_name} ───')
    dfPy04.groupBy(col_name).count() \
          .orderBy(col_name) \
          .show(truncate=False)

In [ ]:
from pyspark.sql import functions as F

nulos = dfPy03.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in dfPy03.columns
])

# Transponer para mejor legibilidad
nulos_lista = [(c, nulos.first()[c]) for c in dfPy03.columns]
nulos_lista.sort(key=lambda x: x[1], reverse=True)

print(f'{"Variable":<35} {"Nulos":>10}')
print('─' * 47)
for col_name, n in nulos_lista:
    if n > 0:
        print(f'{col_name:<35} {n:>10,}')

print('─' * 47)
sin_nulos = sum(1 for _, n in nulos_lista if n == 0)
print(f'Variables sin nulos: {sin_nulos} de {len(dfPy03.columns)}')